In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Given a 1D array <code>A</code> of <code>N</code> 32-bit floating point numbers, compact all
  positive elements (<code>A[i] &gt; 0</code>) to the front of the output array <code>out</code>,
  preserving their original relative order. Fill any remaining positions with <code>0.0</code>.
  Stream compaction is a fundamental GPU primitive used throughout rendering, sparse computation,
  and collision detection.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native GPU features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>
    The first <em>k</em> positions of <code>out</code> must contain the <em>k</em> elements of
    <code>A</code> where <code>A[i] &gt; 0</code>, in their original order
  </li>
  <li>Positions <em>k</em> through <em>N&minus;1</em> of <code>out</code> must be <code>0.0</code></li>
  <li>Elements where <code>A[i] = 0.0</code> are <strong>not</strong> selected</li>
</ul>

<h2>Example</h2>
<pre>
Input:  A = [1.0, -2.0, 3.0, 0.0, -1.0, 4.0]
Output: out = [1.0, 3.0, 4.0, 0.0, 0.0, 0.0]
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>N</code> &le; 100,000,000</li>
  <li>&minus;1000.0 &le; <code>A[i]</code> &le; 1000.0</li>
  <li><code>out</code> is pre-allocated with <code>N</code> elements, initialised to <code>0.0</code></li>
  <li>Performance is measured with <code>N</code> = 50,000,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// A, out are device pointers
extern "C" void solve(const float* A, int N, float* out) {}


# CUTE

In [ ]:
%%writefile solution_cute.py
import cutlass
import cutlass.cute as cute


# A, out are tensors on the GPU
@cute.jit
def solve(A: cute.Tensor, N: cute.Uint32, out: cute.Tensor):
    pass


# JAX

In [ ]:
%%writefile solution_jax.py
import jax
import jax.numpy as jnp


# A is a tensor on GPU
@jax.jit
def solve(A: jax.Array, N: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.memory import UnsafePointer


# A, out are device pointers
@export
def solve(
    A: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
    out: UnsafePointer[Float32, MutExternalOrigin],
) raises:
    pass


# Torch

In [ ]:
%%writefile solution_pytorch.py
import torch


# A, out are tensors on the GPU
def solve(A: torch.Tensor, N: int, out: torch.Tensor):
    pass


# Triton

In [ ]:
%%writefile solution_triton.py
import torch
import triton
import triton.language as tl


# A, out are tensors on the GPU
def solve(A: torch.Tensor, N: int, out: torch.Tensor):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/medium/72_stream_compaction/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch, EVAL_LANG)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
